# Hypothetical Prompt Embeddings (HyPE)

## Overview

HyPE is the **reverse** of HyDE:

| Technique | When it generates | What it generates | Search type |
|---|---|---|---|
| **HyDE** | At **query time** | A hypothetical *answer document* | document → document |
| **HyPE** | At **indexing time** | Hypothetical *questions* per chunk | **question → question** |

Instead of embedding raw text chunks, HyPE generates multiple hypothetical questions for each chunk during indexing. At query time, the user's question is matched against these precomputed questions — turning retrieval into a **question-to-question matching** problem, which is much more natural.

## Benefits

- **No query-time overhead** — all question generation happens offline during indexing
- **Better alignment** — questions match questions better than questions match document chunks
- **Multiple representations per chunk** — each chunk is indexed by several questions, increasing recall

## Models Used

- **LLM**: `gemma3:12b` via Ollama — generates hypothetical questions
- **Embeddings**: `mxbai-embed-large:335m` via Ollama

<div style="text-align: center;">
<img src="./images/hype.svg" alt="HyPE" style="width:70%; height:auto;">
</div>

---
## Step 0: Import Packages

In [1]:
import faiss
from tqdm import tqdm
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama
from langchain_ollama.embeddings import OllamaEmbeddings

---
## Step 1: Set Up LLM and Embedding Model

In [2]:
llm = ChatOllama(temperature=0, model="gemma3:12b")
embedding_model = OllamaEmbeddings(model="mxbai-embed-large:335m")

print("LLM and embedding model ready")

LLM and embedding model ready


---
## Step 2: Load PDF and Split into Chunks

In [3]:
PATH = "data/Understanding_Climate_Change.pdf"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

loader = PyPDFLoader(PATH)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, length_function=len
)
chunks = text_splitter.split_documents(documents)

# Clean chunks: replace tab characters with spaces
for chunk in chunks:
    chunk.page_content = chunk.page_content.replace("\t", " ")

print(f"Loaded {len(documents)} pages, split into {len(chunks)} chunks")
print(f"Chunk size: {CHUNK_SIZE}, overlap: {CHUNK_OVERLAP}")

Loaded 33 pages, split into 97 chunks
Chunk size: 1000, overlap: 200


---
## Step 3: Define the Question Generation Prompt

This is the **core of HyPE**: for each chunk, we ask the LLM to generate several questions that the chunk could answer. These questions become the chunk's representation in the vector store.

In [4]:
question_gen_prompt = PromptTemplate.from_template(
    "Analyze the input text and generate essential questions that, when answered, "
    "capture the main points of the text. Each question should be one line, "
    "without numbering or prefixes.\n\n"
    "Text:\n{chunk_text}\n\nQuestions:\n"
)

question_chain = question_gen_prompt | llm | StrOutputParser()

print("Question generation chain ready")

Question generation chain ready


---
## Step 4: Preview — Generate Questions for One Chunk

Before processing all chunks, let's see what questions the LLM generates for a single chunk.

In [5]:
sample_chunk = chunks[0].page_content
print(f"Chunk text (first 200 chars): {sample_chunk[:200]}...\n")

sample_questions_raw = question_chain.invoke({"chunk_text": sample_chunk})
sample_questions = [q.strip() for q in sample_questions_raw.replace("\n\n", "\n").split("\n") if q.strip()]

print(f"Generated {len(sample_questions)} questions:")
for q in sample_questions:
    print(f"  - {q}")

Chunk text (first 200 chars): Understanding Climate Change 
Chapter 1: Introduction to Climate Change 
Climate change refers to significant, long-term changes in the global climate. The term 
"global climate" encompasses the plane...

Generated 5 questions:
  - What is climate change, in its broadest definition?
  - What factors have significantly contributed to climate change in the past century?
  - How has Earth's climate changed naturally throughout history?
  - What caused the cycles of glacial advance and retreat?
  - When did the modern climate era and human civilization begin?


---
## Step 5: Generate Questions and Build the FAISS Vector Store

Now we process **all chunks**:
1. For each chunk, generate hypothetical questions
2. Embed each question
3. Store the question embedding in FAISS, linked to the original chunk text

Each chunk gets stored **multiple times** — once per generated question. This gives each chunk multiple "entry points" in the embedding space.

This step takes a while since it calls the LLM + embedding model for every chunk.

In [6]:
vector_store = None

for i, chunk in enumerate(tqdm(chunks, desc="Processing chunks")):
    # Generate questions for this chunk
    questions_raw = question_chain.invoke({"chunk_text": chunk.page_content})
    questions = [q.strip() for q in questions_raw.replace("\n\n", "\n").split("\n") if q.strip()]

    if not questions:
        continue

    # Embed all questions for this chunk
    question_vectors = embedding_model.embed_documents(questions)

    # Initialize FAISS on the first chunk (we need to know the vector dimension)
    if vector_store is None:
        vector_store = FAISS(
            embedding_function=embedding_model,
            index=faiss.IndexFlatL2(len(question_vectors[0])),
            docstore=InMemoryDocstore(),
            index_to_docstore_id={}
        )

    # Store each question embedding paired with the original chunk text
    chunks_with_vectors = [(chunk.page_content, vec) for vec in question_vectors]
    vector_store.add_embeddings(chunks_with_vectors)

print(f"\nVector store built with {vector_store.index.ntotal} question embeddings from {len(chunks)} chunks")

Processing chunks: 100%|███████████| 97/97 [17:44<00:00, 10.98s/it]


Vector store built with 556 question embeddings from 97 chunks


---
## Step 6: Create the Retriever and Test

Now when we query, our question is matched against the **precomputed hypothetical questions** — not the raw chunk text. This is question-to-question matching.

In [8]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

test_query = "What is the main cause of climate change?"
print(f"Query: {test_query}\n")

results = retriever.invoke(test_query)

# Deduplicate (multiple question embeddings can point to the same chunk)
seen = set()
unique_results = []
for doc in results:
    if doc.page_content not in seen:
        seen.add(doc.page_content)
        unique_results.append(doc)

print(f"Retrieved {len(results)} results ({len(unique_results)} unique chunks):\n")
for i, doc in enumerate(unique_results, 1):
    print(f"--- Context {i} ---")
    print(doc.page_content)
    print()

Query: What is the main cause of climate change?

Retrieved 3 results (3 unique chunks):

--- Context 1 ---
Chapter 2: Causes of Climate Change 
Greenhouse Gases 
The primary cause of recent climate change is the increase in greenhouse gases in the 
atmosphere. Greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and nitrous 
oxide (N2O), trap heat from the sun, creating a "greenhouse effect." This effect is essential 
for life on Earth, as it keeps the planet warm enough to support life. However, human 
activities have intensified this natural process, leading to a warmer climate. 
Fossil Fuels 
Burning fossil fuels for energy releases large amounts of CO2. This includes coal, oil, and 
natural gas used for electricity, heating, and transportation. The industrial revolution marked 
the beginning of a significant increase in fossil fuel consumption, which continues to rise 
today. 
Coal

--- Context 2 ---
Most of these climate changes are attributed to very small variations i

---
## Summary

| Step | What happened |
|---|---|
| 1 | Set up LLM + embeddings |
| 2 | Loaded PDF, chunked into overlapping segments |
| 3 | Defined a prompt that generates questions from chunk text |
| 4 | Previewed question generation on one chunk |
| 5 | **For every chunk**: generated questions → embedded questions → stored in FAISS (linked to original chunk) |
| 6 | Queried with a natural question → matched against precomputed questions → retrieved original chunks |

**Key insight:** HyPE flips the HyDE approach. Instead of generating a hypothetical *answer* at query time, it generates hypothetical *questions* at indexing time. The result is:
- **Zero overhead at query time** — retrieval is as fast as standard RAG
- **Better semantic matching** — questions match questions better than questions match document paragraphs
- **Multiple entry points** — each chunk is reachable through several different question phrasings